In [15]:
import pandas as pd

df_old = pd.read_csv("data/processed/products_clean_v1.csv")
df_new = pd.read_csv("data/processed/myntra_final_clean.csv")

In [16]:
print("OLD DATASET COLUMNS:\n", df_old.columns)
print("\nNEW DATASET COLUMNS:\n", df_new.columns)

OLD DATASET COLUMNS:
 Index(['product_id', 'title', 'description', 'category', 'brand', 'price',
       'gender', 'color', 'image_path', 'category_clean', 'price_range',
       'tags', 'search_text', 'rating', 'reviews'],
      dtype='object')

NEW DATASET COLUMNS:
 Index(['product_id', 'title', 'description', 'category', 'brand', 'price',
       'gender', 'color', 'image_path', 'category_clean', 'price_range',
       'tags', 'search_text', 'rating', 'reviews'],
      dtype='object')


In [17]:
# Convert both datasets to SAME types

for df in [df_old, df_new]:
    df["product_id"] = df["product_id"].astype(int)
    df["price"] = df["price"].astype(int)
    df["rating"] = df["rating"].astype(float)
    df["reviews"] = df["reviews"].astype(str)

    # Optional but safe
    df["title"] = df["title"].astype(str)
    df["description"] = df["description"].astype(str)
    df["brand"] = df["brand"].astype(str)
    df["color"] = df["color"].astype(str)
    df["image_path"] = df["image_path"].astype(str)

In [18]:
missing_in_old = set(df_new.columns) - set(df_old.columns)
missing_in_new = set(df_old.columns) - set(df_new.columns)

print("Missing in OLD:", missing_in_old)
print("Missing in NEW:", missing_in_new)

Missing in OLD: set()
Missing in NEW: set()


In [19]:
df_new = df_new[df_old.columns]

In [20]:
df_final = pd.concat([df_old, df_new], ignore_index=True)

In [21]:
df_final = df_final.drop_duplicates(subset=["product_id"])

In [22]:
print("Final dataset size:", len(df_final))

Final dataset size: 121710


In [23]:
df_old.to_csv("data/processed/products_backup.csv", index=False)

In [24]:
df_final.to_csv("data/processed/products_clean_v1.csv", index=False)

In [25]:
dups = df_final.duplicated(subset=["product_id"]).sum()
print("Duplicate product_id:", dups)

Duplicate product_id: 0


In [26]:
import os
missing = df_final[~df_final["image_path"].apply(os.path.exists)]
print("Missing images:", len(missing))

Missing images: 26846


In [28]:
df_final["is_url"] = df_final["image_path"].str.startswith("http", na=False)
print("URL images:", df_final["is_url"].sum())

URL images: 26846


In [29]:
import os

df_final["is_local"] = df_final["image_path"].apply(lambda x: os.path.exists(x) if isinstance(x, str) else False)
print("Local images:", df_final["is_local"].sum())

Local images: 94864


In [30]:
import os

def fix_image_path(row):
    local_path = f"images/{int(row['product_id'])}.jpg"
    
    # Prefer local image if exists
    if os.path.exists(local_path):
        return local_path
    
    # Else keep URL
    elif isinstance(row["image_path"], str) and row["image_path"].startswith("http"):
        return row["image_path"]
    
    # Else keep original (don't delete)
    return row["image_path"]

df_final["image_path"] = df_final.apply(fix_image_path, axis=1)

In [31]:
df_final = df_final[
    df_final["image_path"].notna() &
    (df_final["image_path"] != "")
]

In [32]:
df_final.to_csv("data/processed/products_clean_v1.csv", index=False)